## 09 — Building density / size vs F1 analysis

Examines whether reference building density and mean building size predict
per-tile F1 for each candidate dataset.

**Runs three times per data type** (all cities / SpaceNet reference only / non-SpaceNet)
for both vector and raster candidates.

**Inputs:**
- `outputs/scratch/per_tile_enriched_all_cities.csv` (vector, from nb07)
- `outputs/metrics/*/raster_metrics_tiles_all_datasets.parquet` (raster)

**Outputs:** figures in `outputs/figures/`, summary CSV in `outputs/scratch/`.

In [ ]:
!pip install -q scikit-posthocs statsmodels
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup & load ─────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as smf
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

FIGURES_DIR  = PROJECT_ROOT / 'outputs' / 'figures'
SCRATCH_DIR  = PROJECT_ROOT / 'outputs' / 'scratch'
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DENSITY_COL = 'ref_building_density_per_km2'
SIZE_COL    = 'mean_ref_building_area_m2'
F1_COL      = 'f1'

# Okabe-Ito 7-colour CVD-safe palette; fixed vector assignments, remainder dynamic
OKABE_ITO = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#56B4E9', '#D55E00', '#F0E442']
KNOWN_COLORS = {'overture': '#0072B2', 'gba': '#E69F00', 'globfp': '#009E73',
                'Overture': '#0072B2', 'GBA': '#E69F00', 'GlobFP': '#009E73'}

def assign_colors(dataset_names):
    """Assign Okabe-Ito colours; known vector names keep fixed slots."""
    used = set(KNOWN_COLORS.values())
    free = [c for c in OKABE_ITO if c not in used]
    free_iter = iter(free)
    return {ds: KNOWN_COLORS.get(ds, next(free_iter, '#999999'))
            for ds in sorted(dataset_names)}

# ── Load vector enriched CSV ──────────────────────────────────────────────────
df_vec = pd.read_csv(SCRATCH_DIR / 'per_tile_enriched_all_cities.csv')
DS_LABELS_VEC = {'overture': 'Overture', 'gba': 'GBA', 'globfp': 'GlobFP'}
df_vec['dataset_label'] = df_vec['dataset'].str.lower().map(DS_LABELS_VEC).fillna(df_vec['dataset'])
df_vec_clean = df_vec.dropna(subset=[DENSITY_COL, SIZE_COL, F1_COL]).copy()

print('=== Vector enriched CSV ===')
print(f'  Rows after NaN drop : {len(df_vec_clean):,}  ({len(df_vec):,} total)')
print(f'  Cities              : {df_vec_clean["city"].nunique()}')
print(f'  Datasets            : {sorted(df_vec_clean["dataset"].str.lower().unique())}')

In [ ]:
# ── Cell 2 — Reference-source detection (SpaceNet vs other) ──────────────────
# Reads the AOI tracker CSV; any city whose reference filename contains
# "spacenet" (case-insensitive) is tagged ref_source = 'spacenet'.

tracker_path = Path(cfg.get('aoi_tracker', 'data/02_interim/aoi_tracker.csv'))
if not tracker_path.is_absolute():
    tracker_path = PROJECT_ROOT / tracker_path

ref_source_map = {}  # city_id -> 'spacenet' | 'other'

if tracker_path.exists():
    tracker = pd.read_csv(tracker_path, dtype=str)
    tracker.columns = tracker.columns.str.strip()
    id_col  = 'dataset_folder_name'
    ref_col = next((c for c in tracker.columns
                    if 'reference' in c.lower() and 'file' in c.lower()), None)
    if ref_col and id_col in tracker.columns:
        city_ref = (
            tracker.groupby(id_col)[ref_col]
            .apply(lambda x: '|'.join(x.dropna().str.lower()))
            .reset_index()
            .rename(columns={id_col: 'city', ref_col: 'ref_files'})
        )
        city_ref['ref_source'] = city_ref['ref_files'].apply(
            lambda x: 'spacenet' if 'spacenet' in str(x) else 'other'
        )
        ref_source_map = city_ref.set_index('city')['ref_source'].to_dict()
        print(f'Tracker loaded: {len(ref_source_map)} cities, ref_col = "{ref_col}"')
    else:
        print(f'[WARN] ref_col not found (ref_col={ref_col}) — defaulting all to "other"')
else:
    print(f'[WARN] Tracker not found at {tracker_path} — defaulting all to "other"')

def tag_ref_source(df):
    df = df.copy()
    df['ref_source'] = df['city'].map(ref_source_map).fillna('other')
    return df

df_vec_clean = tag_ref_source(df_vec_clean)

n_sn  = df_vec_clean['city'][df_vec_clean['ref_source'] == 'spacenet'].nunique()
n_oth = df_vec_clean['city'][df_vec_clean['ref_source'] == 'other'].nunique()
print(f'\n  SpaceNet cities : {n_sn}')
print(f'  Other cities    : {n_oth}')
if n_sn:
    sn_list = sorted(df_vec_clean[df_vec_clean['ref_source'] == 'spacenet']['city'].unique())
    print(f'  SpaceNet city list : {sn_list}')

def make_subsets(df):
    return {
        'all':          df,
        'non_spacenet': df[df['ref_source'] == 'other'].copy(),
        'spacenet':     df[df['ref_source'] == 'spacenet'].copy(),
    }

VEC_SUBSETS = make_subsets(df_vec_clean)
print('\n  Subset sizes (vector):')
for k, v in VEC_SUBSETS.items():
    print(f'    {k:<15}: {len(v):>6,} rows | {v["city"].nunique()} cities')

In [ ]:
# ── Cell 3 — Analysis helper functions ───────────────────────────────────────
# Defined once; called for every (data_type × subset) combination.

def global_quartile_bins(df, col=DENSITY_COL):
    """Compute quartile boundaries from the full dataset.
    Applying the same bins to subsets keeps Q1-Q4 definitions consistent.
    """
    q = df[col].quantile([0, 0.25, 0.5, 0.75, 1.0]).values
    labels = [
        f'Q1\n(<{q[1]:.0f})',
        f'Q2\n({q[1]:.0f}–{q[2]:.0f})',
        f'Q3\n({q[2]:.0f}–{q[3]:.0f})',
        f'Q4\n(>{q[3]:.0f})',
    ]
    return q, labels


def add_density_q(df, bins, labels):
    df = df.copy()
    df['density_q'] = pd.cut(df[DENSITY_COL], bins=bins, labels=labels, include_lowest=True)
    return df


def _section_header(title, group_label, n_rows, n_cities):
    bar = '─' * 60
    print(f'\n{bar}')
    print(f'  {title}  |  group: {group_label}  ({n_rows:,} rows, {n_cities} cities)')
    print(bar)


def run_spearman(df, group_label):
    """Spearman ρ per dataset for density→F1 and size→F1."""
    _section_header('Spearman ρ (marginal)', group_label, len(df), df['city'].nunique())
    results = []
    for ds in sorted(df['dataset'].str.lower().unique()):
        sub = df[df['dataset'].str.lower() == ds]
        label = DS_LABELS_VEC.get(ds, ds)
        rho_d, p_d = stats.spearmanr(sub[DENSITY_COL], sub[F1_COL])
        rho_s, p_s = stats.spearmanr(sub[SIZE_COL],    sub[F1_COL])
        sig_d = '***' if p_d < 0.001 else ('**' if p_d < 0.01 else ('*' if p_d < 0.05 else 'ns'))
        sig_s = '***' if p_s < 0.001 else ('**' if p_s < 0.01 else ('*' if p_s < 0.05 else 'ns'))
        print(f'{label} (n={len(sub):,})')
        print(f'  density→F1: ρ = {rho_d:+.3f}  p = {p_d:.2e}  {sig_d}')
        print(f'  size→F1:    ρ = {rho_s:+.3f}  p = {p_s:.2e}  {sig_s}')
        results.append(dict(group=group_label, dataset=label, n=len(sub),
                            rho_density=rho_d, p_density=p_d,
                            rho_size=rho_s, p_size=p_s))
    return results


def run_kw(df, group_label, q_labels):
    """Kruskal–Wallis + Dunn post-hoc per dataset."""
    _section_header('Kruskal–Wallis + Dunn (Bonferroni)', group_label, len(df), df['city'].nunique())
    results = []
    for ds in sorted(df['dataset'].str.lower().unique()):
        sub   = df[df['dataset'].str.lower() == ds].dropna(subset=['density_q'])
        label = DS_LABELS_VEC.get(ds, ds)
        groups = [g[F1_COL].values for _, g in sub.groupby('density_q', observed=True) if len(g) > 0]
        if len(groups) < 2:
            print(f'{label}: insufficient groups — skipped')
            continue
        H, p = stats.kruskal(*groups)
        sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f'{label}: H = {H:.2f}  p = {p:.2e}  {sig}')
        if p < 0.05:
            dunn = sp.posthoc_dunn(sub, val_col=F1_COL, group_col='density_q', p_adjust='bonferroni')
            print(dunn.round(4).to_string())
        results.append(dict(group=group_label, dataset=label, H=H, p=p, sig=sig))
    return results


def plot_boxplot(df, group_label, q_labels, color_map, save_tag, figures_dir):
    """Box plot: F1 by density quartile, one panel per dataset."""
    ds_order = [d for d in sorted(color_map) if d in df['dataset_label'].unique()]
    if not ds_order:
        print(f'  [SKIP] no datasets for boxplot ({group_label})')
        return
    fig, axes = plt.subplots(1, len(ds_order), figsize=(4.5 * len(ds_order), 5), sharey=True)
    if len(ds_order) == 1:
        axes = [axes]
    for ax, ds_label in zip(axes, ds_order):
        sub   = df[df['dataset_label'] == ds_label].dropna(subset=['density_q'])
        color = color_map.get(ds_label, '#666666')
        sns.boxplot(data=sub, x='density_q', y=F1_COL, ax=ax, order=q_labels,
                    color=color, width=0.55, linewidth=1.2,
                    flierprops=dict(marker='o', markersize=2, alpha=0.25,
                                   markerfacecolor=color, markeredgecolor='none'),
                    medianprops=dict(color='white', linewidth=2))
        for i, ql in enumerate(q_labels):
            n = (sub['density_q'] == ql).sum()
            ax.text(i, -0.06, f'n={n:,}', ha='center', va='top', fontsize=7.5,
                    color='#555555', transform=ax.get_xaxis_transform())
        ax.set_title(ds_label, fontsize=12, fontweight='bold', color=color, pad=8)
        ax.set_xlabel('Density quartile (bldg/km²)', fontsize=9)
        ax.set_ylabel('F1 score' if ax == axes[0] else '', fontsize=9)
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
        ax.grid(axis='y', color='#e0e0e0', linewidth=0.7, zorder=0)
        ax.set_axisbelow(True)
        sns.despine(ax=ax)
    fig.suptitle(f'F1 by density quartile — {group_label}', fontsize=13, y=1.01)
    fig.tight_layout()
    path = figures_dir / f'f1_by_density_quartile_{save_tag}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved → {path}')


def plot_scatters(df, group_label, color_map, save_tag, figures_dir):
    """Scatter: (a) density vs F1 (log-x), (b) mean size vs F1. Coloured by dataset."""
    ds_order = [d for d in sorted(color_map) if d in df['dataset_label'].unique()]

    def _scatter(ax, x_col, x_label, log_x=False):
        for ds_label in ds_order:
            sub = df[df['dataset_label'] == ds_label]
            ax.scatter(sub[x_col], sub[F1_COL], color=color_map.get(ds_label, '#666'),
                       alpha=0.3, s=8, linewidths=0, label=ds_label, rasterized=True)
        if log_x:
            ax.set_xscale('log')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel('F1 score', fontsize=10)
        ax.set_ylim(-0.02, 1.05)
        ax.grid(color='#e8e8e8', linewidth=0.6, zorder=0)
        ax.set_axisbelow(True)
        ax.legend(title='Dataset', fontsize=8, title_fontsize=8,
                  markerscale=3, framealpha=0.85)
        sns.despine(ax=ax)

    for x_col, x_label, log_x, fname in [
        (DENSITY_COL, 'Reference building density (bldg/km²)', True,  f'density_f1_scatter_{save_tag}.png'),
        (SIZE_COL,    'Mean reference building area (m²)',      False, f'size_f1_scatter_{save_tag}.png'),
    ]:
        fig, ax = plt.subplots(figsize=(7, 5))
        _scatter(ax, x_col, x_label, log_x)
        ax.set_title(f'{x_label.split("(")[0].strip()} vs F1 — {group_label}', fontsize=12)
        fig.tight_layout()
        path = figures_dir / fname
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  Saved → {path}')


def run_lme(df, group_label):
    """Linear mixed-effects model: f1 ~ density_z + size_z + C(dataset), city random intercept."""
    _section_header('Linear mixed-effects model', group_label, len(df), df['city'].nunique())
    dfl = df[['city', 'dataset', F1_COL, DENSITY_COL, SIZE_COL]].dropna().copy()
    dfl['density_z'] = (dfl[DENSITY_COL] - dfl[DENSITY_COL].mean()) / dfl[DENSITY_COL].std()
    dfl['size_z']    = (dfl[SIZE_COL]    - dfl[SIZE_COL].mean())    / dfl[SIZE_COL].std()
    dfl['dataset']   = dfl['dataset'].str.lower()
    if dfl['city'].nunique() < 2:
        print('  [SKIP] fewer than 2 cities — LME cannot fit random intercept')
        return None
    result = smf.mixedlm('f1 ~ density_z + size_z + C(dataset)', data=dfl,
                         groups=dfl['city']).fit(reml=True)
    print(result.summary())
    print('\n  Key fixed effects:')
    for name, label in [('density_z', 'density (z)'), ('size_z', 'mean size (z)')]:
        coef = result.params[name]
        pval = result.pvalues[name]
        ci   = result.conf_int().loc[name]
        sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        print(f'  {label:<20}: β = {coef:+.4f}  [{ci[0]:+.4f}, {ci[1]:+.4f}]  p = {pval:.2e}  {sig}')
    print(f'  Random-effect var (city): {result.cov_re.iloc[0,0]:.4f}')
    return result


def run_group(df, group_label, q_bins, q_labels, color_map, save_tag, figures_dir):
    """Orchestrate all analyses for one (data_type, subset) combination."""
    print(f'\n{"="*65}')
    print(f'  GROUP: {group_label}  ({len(df):,} rows | {df["city"].nunique()} cities)')
    print(f'{"="*65}')
    if len(df) < 10:
        print('  [SKIP] fewer than 10 rows')
        return {}
    df_q = add_density_q(df, q_bins, q_labels)
    s    = run_spearman(df_q, group_label)
    kw   = run_kw(df_q, group_label, q_labels)
    plot_boxplot(df_q, group_label, q_labels, color_map, save_tag, figures_dir)
    plot_scatters(df_q, group_label, color_map, save_tag, figures_dir)
    lme  = run_lme(df_q, group_label)
    return dict(spearman=s, kw=kw, lme=lme)


print('Analysis functions defined.')

---
## Vector analysis
Candidates: Overture, GBA, GlobFP — compared against local reference buildings.

In [ ]:
# ── Cell 4 — Vector analysis (all 3 subsets) ──────────────────────────────────
VEC_Q_BINS, VEC_Q_LABELS = global_quartile_bins(df_vec_clean)  # from full dataset
VEC_DS_ALL   = sorted(df_vec_clean['dataset_label'].unique())
VEC_COLOR_MAP = assign_colors(VEC_DS_ALL)

print('Global vector quartile boundaries (bldg/km²):')
for i, (lo, hi) in enumerate(zip(VEC_Q_BINS[:-1], VEC_Q_BINS[1:])):
    print(f'  Q{i+1}: {lo:.1f} – {hi:.1f}')
print(f'Dataset colour map: {VEC_COLOR_MAP}')

vec_results = {}
for group_label, sdf in VEC_SUBSETS.items():
    save_tag = f'vector_{group_label}'
    vec_results[group_label] = run_group(
        sdf, group_label,
        VEC_Q_BINS, VEC_Q_LABELS,
        VEC_COLOR_MAP, save_tag, FIGURES_DIR,
    )

---
## Raster analysis
Raster candidates (GHSL, WSF, …) compared against the same reference buildings.
Density columns are joined from the vector enriched CSV (same tile grid).

In [ ]:
# ── Cell 5 — Load raster data & join density ──────────────────────────────────
# Raster sentinel: outputs/metrics/{city}/raster_metrics_tiles_all_datasets.parquet
# Has columns: tile_id, city, dataset, grid, resolution_m, f1, precision, recall, …
# One row per (tile_id, dataset, grid, resolution_m).

RASTER_SENTINEL = 'raster_metrics_tiles_all_datasets.parquet'

# If multiple evaluation grids exist, set RASTER_GRID to filter to one;
# None = average f1 across all grids per (city, tile_id, dataset).
RASTER_GRID = None   # e.g. 'native'  or  None

raster_parts = []
raster_city_dirs = sorted(p.parent for p in METRICS_ROOT.rglob(RASTER_SENTINEL))
print(f'Found {len(raster_city_dirs)} cities with raster metrics')

for city_dir in raster_city_dirs:
    try:
        part = pd.read_parquet(city_dir / RASTER_SENTINEL)
        raster_parts.append(part)
    except Exception as e:
        print(f'  [WARN] {city_dir.name}: {e}')

if not raster_parts:
    print('[INFO] No raster metrics found. Raster analysis will be skipped.')
    df_rast_clean = pd.DataFrame(columns=['city', 'tile_id', 'dataset', 'dataset_label',
                                           F1_COL, DENSITY_COL, SIZE_COL, 'ref_source'])
else:
    df_rast = pd.concat(raster_parts, ignore_index=True)
    print(f'Raster rows loaded: {len(df_rast):,}')
    print(f'Evaluation grids found: {sorted(df_rast["grid"].unique()) if "grid" in df_rast.columns else "no grid column"}')
    print(f'Raster datasets: {sorted(df_rast["dataset"].unique())}')

    # Filter or aggregate across evaluation grids
    if 'grid' in df_rast.columns:
        if RASTER_GRID is not None:
            df_rast = df_rast[df_rast['grid'] == RASTER_GRID].copy()
            print(f'Filtered to grid "{RASTER_GRID}": {len(df_rast):,} rows')
        else:
            agg_cols = {F1_COL: 'mean', 'precision': 'mean', 'recall': 'mean'}
            agg_cols = {k: v for k, v in agg_cols.items() if k in df_rast.columns}
            keep_cols = ['city', 'tile_id', 'dataset'] + list(agg_cols)
            df_rast = df_rast[keep_cols].groupby(['city', 'tile_id', 'dataset']).agg(agg_cols).reset_index()
            print(f'Averaged across grids: {len(df_rast):,} rows')

    # Join density columns from vector enriched CSV (same tile grid)
    density_lookup = (
        df_vec_clean[['city', 'tile_id', DENSITY_COL, SIZE_COL,
                       'tile_area_km2', 'ref_building_count_centroid']]
        .drop_duplicates(subset=['city', 'tile_id'])
    )
    df_rast = df_rast.merge(density_lookup, on=['city', 'tile_id'], how='left')
    df_rast['dataset_label'] = df_rast['dataset']  # raster names used as-is
    df_rast_clean = df_rast.dropna(subset=[DENSITY_COL, SIZE_COL, F1_COL]).copy()
    df_rast_clean = tag_ref_source(df_rast_clean)

    joined_pct = len(df_rast_clean) / len(df_rast) * 100 if len(df_rast) else 0
    print(f'After density join + NaN drop: {len(df_rast_clean):,} rows ({joined_pct:.1f}%)')
    print(f'Cities: {df_rast_clean["city"].nunique()}')
    print()
    if len(df_rast_clean) < len(df_rast) * 0.5:
        print('[WARN] > 50% of raster rows dropped — many raster cities may lack'
              ' density enrichment (tiles GPKG missing). Results cover a partial sample.')

In [ ]:
# ── Cell 6 — Raster analysis (all 3 subsets) ──────────────────────────────────
if df_rast_clean.empty:
    print('No raster data — skipping.')
else:
    RAST_Q_BINS, RAST_Q_LABELS = global_quartile_bins(df_rast_clean)
    RAST_DS_ALL    = sorted(df_rast_clean['dataset_label'].unique())
    RAST_COLOR_MAP = assign_colors(RAST_DS_ALL)
    RAST_SUBSETS   = make_subsets(df_rast_clean)

    print('Global raster quartile boundaries (bldg/km²):')
    for i, (lo, hi) in enumerate(zip(RAST_Q_BINS[:-1], RAST_Q_BINS[1:])):
        print(f'  Q{i+1}: {lo:.1f} – {hi:.1f}')
    print(f'Raster dataset colour map: {RAST_COLOR_MAP}')
    print('\n  Subset sizes (raster):')
    for k, v in RAST_SUBSETS.items():
        print(f'    {k:<15}: {len(v):>6,} rows | {v["city"].nunique()} cities')

    # Raster helper: dataset_label lookup differs from vector, so patch run_spearman
    rast_results = {}
    for group_label, sdf in RAST_SUBSETS.items():
        save_tag = f'raster_{group_label}'
        rast_results[group_label] = run_group(
            sdf, group_label,
            RAST_Q_BINS, RAST_Q_LABELS,
            RAST_COLOR_MAP, save_tag, FIGURES_DIR,
        )

In [ ]:
# ── Cell 7 — Limitation note & save ──────────────────────────────────────────
print('=== Analysis limitations ===')
print()
print('  Spearman ρ is marginal (tiles nested in cities — city confounding not removed).')
print('  Treat as exploratory. The LME random intercept removes city-level confounding.')
print()
print('  SpaceNet / other split is based on the reference filename in the tracker CSV.')
print('  Verify the SpaceNet city list above matches expectations.')
print()
print('  ~65 cities were excluded from the enriched CSV (missing tiles GPKG).')
print('  Raster cities lacking a density join are also dropped — check the join % above.')
print()
print('  Density quartile boundaries are computed from the FULL dataset (all groups)')  
print('  so Q1–Q4 thresholds are identical across all subsets and data types.')
print()

# Save analysis inputs for reproducibility
vec_out = SCRATCH_DIR / 'density_analysis_input_vector.csv'
df_vec_clean.to_csv(vec_out, index=False)
print(f'Vector analysis input saved → {vec_out}')

if 'df_rast_clean' in dir() and not df_rast_clean.empty:
    rast_out = SCRATCH_DIR / 'density_analysis_input_raster.csv'
    df_rast_clean.to_csv(rast_out, index=False)
    print(f'Raster analysis input saved → {rast_out}')

print()
print('=== Figure files generated ===')
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(f'  {f.name}')